In [0]:
df = spark.table('gizmobox.bronze.py_payments')
display(df)

In [0]:
df.printSchema()

In [0]:
# Extract payment data and time from payment timestamp and create seperate columns for each

from pyspark.sql.functions import *
df = df.withColumn('payment_date',to_date(col('payment_timestamp'))).withColumn('payment_time',date_format(col('payment_timestamp'), 'HH:mm:ss'))

display(df)

In [0]:
df = df.drop("payment_timestamp")
display(df)

In [0]:
# 1-Success 2-Pending 3-Cancelled-4 Failed

df = df.withColumn('status',when(col('payment_status')==1,'Sucess')
                                    .when(col('payment_status')==2,'Pending')
                                    .when(col("payment_status")==3,'Cancelled')
                                    .otherwise('Failed'))
display(df)


In [0]:
df = df.drop("payment_status").withColumnRenamed('status','payment_status')
display(df)

In [0]:
# writing to silver layer
df.writeTo('gizmobox.silver.py_payments').createOrReplace()